In [1]:
# Creating an exch2 file


In [10]:
import os
import netCDF4 as nc4
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def write_exch2_files(output_file, rows, cols,  blank_list):
    exch2_output = ' &W2_EXCH2_PARM01\n'
    exch2_output += '  W2_mapIO   = 1,\n'
    exch2_output += '  preDefTopol = 1,\n'
    exch2_output += '  dimsFacets = '+str(cols)+', '
    exch2_output += str(rows) + ',\n'
    exch2_output += '#\n'
    exch2_output += '  blankList ='
    for counter in blank_list:
        exch2_output += '  '+str(counter)+',\n'
    exch2_output += ' &'

    f = open(output_file,'w')
    f.write(exch2_output)
    f.close()

In [13]:
input_dir = '/Users/mhwood/Documents/Research/Projects/Chukchi_Sea/Model/input'
namelist_dir = '/Users/mhwood/Documents/Research/Projects/Chukchi_Sea/Model/namelist'

In [14]:
def read_L3_bathy_from_grid(grid_dir, model_name):
    file_path = os.path.join(grid_dir, model_name + '_grid.nc')
    ds = nc4.Dataset(file_path)
    depth = ds.variables['Depth'][:, :]
    ds.close()
    return(depth)

In [15]:
model_name = 'Chukchi_Sea'
print('Creating the data.exch2 file for the '+model_name+' model')

sNx = 60
sNy = 60

depth = read_L3_bathy_from_grid(input_dir, model_name)
rows = np.shape(depth)[0]
cols = np.shape(depth)[1]

n_tile_rows = int(np.shape(depth)[0] / sNy)
n_tile_cols = int(np.shape(depth)[1] / sNx)
n_blank_cells = 0
total_cells = 0
blank_list = []
for j in range(n_tile_rows):
    for i in range(n_tile_cols):
        total_cells += 1
        tile_subset = depth[j * sNy:(j + 1) * sNy, i * sNx:(i + 1) * sNx]
        if not np.any(tile_subset != 0):
            n_blank_cells += 1
            blank_list.append(total_cells)

print('    - Domain size: '+str(np.shape(depth)))
print('    - Tile size: '+str(sNx)+' by '+str(sNy))
print('    - Tile rows / cols: '+str(np.shape(depth)[0] / sNy)+' / '+str(np.shape(depth)[1] / sNx))
print('    - Blank cells: '+str(n_blank_cells))
print('    - Non-blank cells: '+str(total_cells-n_blank_cells))
print('    - Total cells: '+str(total_cells))
print('    - Proc breakdown: ')
print('        - Ivy procs: ' + str((total_cells - n_blank_cells) / 20))
print('        - Has procs: ' + str((total_cells - n_blank_cells) / 24))
print('        - Bro procs: ' + str((total_cells - n_blank_cells) / 28))
print('    - Blank list: '+str(blank_list))

output_file = os.path.join(namelist_dir, 'data.exch2_'+str(sNx)+'_'+str(sNy))
write_exch2_files(output_file, rows, cols, blank_list)

Creating the data.exch2 file for the Chukchi_Sea model
    - Domain size: (720, 960)
    - Tile size: 60 by 60
    - Tile rows / cols: 12.0 / 16.0
    - Blank cells: 63
    - Non-blank cells: 129
    - Total cells: 192
    - Proc breakdown: 
        - Ivy procs: 6.45
        - Has procs: 5.375
        - Bro procs: 4.607142857142857
    - Blank list: [12, 13, 14, 15, 16, 17, 30, 31, 32, 33, 34, 35, 36, 46, 47, 48, 49, 50, 51, 62, 63, 64, 65, 66, 67, 68, 69, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 94, 95, 96, 97, 98, 99, 100, 101, 102, 108, 109, 110, 111, 112, 113, 114, 116, 125, 126, 127, 128, 129, 141, 142, 143, 144]
